In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_community.vectorstores import Chroma

c:\RAG\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
loaded_data=TextLoader("self.txt",encoding="utf-8")
docs=loaded_data.load()

In [4]:
text_splitters=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n","\n"," ",""]
)

In [5]:
chunks=text_splitters.split_documents(docs)

In [6]:
embeddings=OpenAIEmbeddings()
llm=ChatOpenAI(
    model="gpt-3.5-turbo"
)

In [8]:
persist_directory="./Chromadb"
vectorstores=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_collection"
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retreiver=vectorstores.as_retriever(k=3)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_prompt=ChatPromptTemplate.from_messages([
    ("system","""You are a helpful assistant who solves the query of user by understanding the following context :  \n {context}
     """),
     ("human","{question}")
])

rag_chain=(
    {
    "context":retreiver|format_docs,
    "question":RunnablePassthrough()
    }|
    rag_prompt
    |llm|
    StrOutputParser()
)